In [ ]:
import json
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.stats import gaussian_kde

In [2]:
def load_timeline_items(json_path: str) -> list:
  """Loads a Google Timeline JSON export and returns its list of items.

  Handles both a direct list structure and one nested under a
  'timelineObjects' key.
  """
  with open(json_path, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

  return (
      raw_data
      if isinstance(raw_data, list)
      else raw_data.get("timelineObjects", raw_data)
  )


def extract_coordinates_from_semantic_json(json_path: str) -> pd.DataFrame:
  """Extracts latitude/longitude coordinates from a Google Timeline export.

  Reads 'visit -> topCandidate -> placeLocation' from each timeline item.

  Parameters
  ----------
  json_path : str
      Path to the Google Timeline JSON file.

  Returns
  -------
  pd.DataFrame
      DataFrame with 'latitude' and 'longitude' columns.
  """
  print(f"Loading Timeline dataset from: {json_path}...")
  timeline_items = load_timeline_items(json_path)

  extracted_points = []
  for item in timeline_items:
    visit_data = item.get("visit", {})
    top_candidate = visit_data.get("topCandidate", {})
    place_location = top_candidate.get("placeLocation", "")

    # Parse coordinates formatted as "geo:latitude,longitude"
    if place_location.startswith("geo:"):
      try:
        latitude_str, longitude_str = place_location.replace(
            "geo:", ""
        ).split(",")
        extracted_points.append((float(latitude_str), float(longitude_str)))
      except (ValueError, AttributeError):
        continue

  locations_df = pd.DataFrame(
      extracted_points, columns=["latitude", "longitude"]
  )

  # Guard against malformed coordinates outside valid geographic bounds
  valid_locations_df = locations_df[
      (locations_df["latitude"].between(-90, 90))
      & (locations_df["longitude"].between(-180, 180))
  ].copy()

  print(
      f"Successfully extracted {len(valid_locations_df)} valid location"
      " points."
  )
  return valid_locations_df

In [ ]:
# Set your file path here
TIMELINE_JSON_PATH = "location-history.json"

# Americas bounding box: Patagonia (Argentina) to Alaska (USA), Pacific
# coast to mid-Atlantic. Used throughout the rest of the notebook.
MIN_LON, MAX_LON = -170.0, -30.0
MIN_LAT, MAX_LAT = -60.0, 75.0

# Extract location data
location_data_df = extract_coordinates_from_semantic_json(TIMELINE_JSON_PATH)

# Display the first few rows to verify extraction
location_data_df.head()

In [ ]:
# Identify the principal cities visited within the Americas bbox. Uses
# offline reverse geocoding (most Timeline entries have no usable place
# name), clusters nearby visits into metro areas, and labels each cluster
# with the largest known city nearby rather than the closest small
# locality. Keeps only the most recent visit per city.
import reverse_geocoder as rg
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

METRO_RADIUS_KM = 50.0  # Visits within this range are treated as one place
MAJOR_CITY_POPULATION_THRESHOLD = 100_000  # Min. population to count as a "principal" city
EARTH_RADIUS_KM = 6371.0
GEONAMES_MAJOR_CITIES_URL = "https://download.geonames.org/export/dump/cities15000.zip"

timeline_items = load_timeline_items(TIMELINE_JSON_PATH)

# Collect every visit's coordinates + timestamp within the Americas bbox
visit_records = []
for item in timeline_items:
  visit_data = item.get("visit", {})
  top_candidate = visit_data.get("topCandidate", {})
  place_location = top_candidate.get("placeLocation", "")
  start_time_str = item.get("startTime", "")

  if not place_location.startswith("geo:"):
    continue

  try:
    latitude_val, longitude_val = map(
        float, place_location.replace("geo:", "").split(",")
    )
  except (ValueError, AttributeError):
    continue

  if not (
      MIN_LON <= longitude_val <= MAX_LON and MIN_LAT <= latitude_val <= MAX_LAT
  ):
    continue

  visit_records.append({
      "latitude": latitude_val,
      "longitude": longitude_val,
      "visit_time": pd.to_datetime(start_time_str, errors="coerce", utc=True),
  })

visits_df = pd.DataFrame(visit_records).dropna(subset=["visit_time"])
print(f"Collected {len(visits_df)} visit points within the Americas bbox.")


def to_unit_sphere_xyz(lat_deg: np.ndarray, lon_deg: np.ndarray) -> np.ndarray:
  """Projects lat/lon degrees onto a unit sphere for Euclidean distance
  queries (cKDTree doesn't support great-circle distance directly)."""
  lat_rad, lon_rad = np.radians(lat_deg), np.radians(lon_deg)
  return np.column_stack([
      np.cos(lat_rad) * np.cos(lon_rad),
      np.cos(lat_rad) * np.sin(lon_rad),
      np.sin(lat_rad),
  ])


def km_to_chord_distance(distance_km: float) -> float:
  """Converts a great-circle distance to the equivalent straight-line
  (chord) distance between two points on the unit sphere."""
  return 2 * np.sin((distance_km / EARTH_RADIUS_KM) / 2)


print("Reverse geocoding visit points to their nearest locality (offline lookup)...")
coordinates = list(zip(visits_df["latitude"], visits_df["longitude"]))
# mode=1 forces single-threaded lookup, avoiding multiprocessing issues
# inside Jupyter kernels
geocode_results = rg.search(coordinates, mode=1)

visits_df["city"] = [result["name"] for result in geocode_results]
visits_df["state"] = [result["admin1"] for result in geocode_results]
visits_df["country"] = [result["cc"] for result in geocode_results]

# --- Cluster nearby visits into metro areas ---------------------------
# Group visit points within METRO_RADIUS_KM of each other so a city and
# its surrounding suburbs are treated as a single visited place.
visit_xyz = to_unit_sphere_xyz(
    visits_df["latitude"].to_numpy(), visits_df["longitude"].to_numpy()
)
metro_eps = km_to_chord_distance(METRO_RADIUS_KM)

visit_tree = cKDTree(visit_xyz)
close_pairs = visit_tree.query_pairs(r=metro_eps, output_type="ndarray")

num_points = len(visits_df)
if len(close_pairs) > 0:
  row_idx = np.concatenate([close_pairs[:, 0], close_pairs[:, 1]])
  col_idx = np.concatenate([close_pairs[:, 1], close_pairs[:, 0]])
  adjacency = coo_matrix(
      (np.ones(len(row_idx)), (row_idx, col_idx)),
      shape=(num_points, num_points),
  )
  _, cluster_labels = connected_components(adjacency, directed=False)
else:
  cluster_labels = np.arange(num_points)

visits_df["metro_cluster"] = cluster_labels

# --- Load a "principal cities" reference so clusters are labeled with the
# largest nearby city instead of the closest small locality -------------
print(
    "Downloading GeoNames major cities database (population >="
    f" {MAJOR_CITY_POPULATION_THRESHOLD:,})..."
)
geonames_columns = [
    "geonameid", "name", "asciiname", "alternatenames", "latitude",
    "longitude", "feature_class", "feature_code", "country_code", "cc2",
    "admin1_code", "admin2_code", "admin3_code", "admin4_code",
    "population", "elevation", "dem", "timezone", "modification_date",
]
major_cities_df = pd.read_csv(
    GEONAMES_MAJOR_CITIES_URL,
    sep="\t",
    header=None,
    names=geonames_columns,
    usecols=["name", "latitude", "longitude", "country_code", "population"],
    compression="zip",
)
major_cities_df = major_cities_df[
    (major_cities_df["population"] >= MAJOR_CITY_POPULATION_THRESHOLD)
    & (major_cities_df["longitude"].between(MIN_LON, MAX_LON))
    & (major_cities_df["latitude"].between(MIN_LAT, MAX_LAT))
].reset_index(drop=True)

# Reverse geocode the major cities themselves so their state/province name
# uses the same readable format as the visit lookups above
major_geocode_results = rg.search(
    list(zip(major_cities_df["latitude"], major_cities_df["longitude"])),
    mode=1,
)
major_cities_df["state"] = [r["admin1"] for r in major_geocode_results]

major_xyz = to_unit_sphere_xyz(
    major_cities_df["latitude"].to_numpy(),
    major_cities_df["longitude"].to_numpy(),
)
major_tree = cKDTree(major_xyz)
print(f"{len(major_cities_df)} major cities loaded for the Americas region.")


def summarize_metro_cluster(group: pd.DataFrame) -> pd.Series:
  """Collapses a cluster of nearby visits into one representative city row.

  Labels the cluster with the largest known city within METRO_RADIUS_KM of
  its centroid; falls back to the most-visited local place name if no
  major city is nearby.
  """
  cluster_centroid_xyz = visit_xyz[group.index.to_numpy()].mean(axis=0)
  cluster_centroid_xyz /= np.linalg.norm(cluster_centroid_xyz)

  nearby_major_idx = major_tree.query_ball_point(
      cluster_centroid_xyz, r=metro_eps
  )
  most_recent_visit = group.sort_values("visit_time", ascending=False).iloc[0]

  if nearby_major_idx:
    nearby_majors = major_cities_df.iloc[nearby_major_idx]
    principal_city = nearby_majors.loc[nearby_majors["population"].idxmax()]
    city_name = principal_city["name"]
    state_name = principal_city["state"]
    country_code = principal_city["country_code"]
  else:
    most_visited_city = group["city"].mode().iloc[0]
    representative = group[group["city"] == most_visited_city].iloc[0]
    city_name = most_visited_city
    state_name = representative["state"]
    country_code = representative["country"]

  return pd.Series({
      "city": city_name,
      "state": state_name,
      "country": country_code,
      "latitude": most_recent_visit["latitude"],
      "longitude": most_recent_visit["longitude"],
      "last_visit": most_recent_visit["visit_time"],
      "visit_count": len(group),
  })


visited_cities_df = (
    visits_df.groupby("metro_cluster")
    .apply(summarize_metro_cluster, include_groups=False)
    .reset_index(drop=True)
    .sort_values("last_visit", ascending=False)
    .reset_index(drop=True)
)

# Export to CSV for downstream use (city markers in the interactive 3D map)
OUTPUT_CITIES_CSV_PATH = "visited_cities.csv"
visited_cities_df.to_csv(OUTPUT_CITIES_CSV_PATH, index=False)

print(
    f"✅ Identified {len(visited_cities_df)} visited cities/metro areas."
    f" Saved to {OUTPUT_CITIES_CSV_PATH}"
)
visited_cities_df.head(20)

In [ ]:
# Interactive 3D relief map (Plotly). Renders visit density -- from
# EVERY visit point, not just the principal cities below -- as a
# realistic-looking terrain: ocean in blue, unvisited land as green
# plains, and visited areas rising into hills/mountains colored by
# elevation (green -> tan -> brown -> white), textured with a bit of
# procedural noise so both the slopes and the outline of each hill read
# as natural terrain rather than a smooth, perfectly oval Gaussian dome.
# The ocean and plains aren't flat solid colors either -- a coastal-depth
# gradient plus subtle mottling gives them the look of a real-world map.
# Country borders are draped over the terrain, and the principal cities
# from the previous step are marked and labeled ("City - year") on top --
# that 50 km clustering is used only to choose which points get a marker
# and a label, not to filter what shapes the terrain. Marker *color*
# separately encodes recency (dark = long ago, bright = recent), with its
# own legend distinct from the terrain's. A title, both color scales, a
# short "how to read this" note, and camera-preset buttons complete the
# UI; a loading overlay covers the initial WebGL setup, and a plain HTML
# table beneath the map -- with a live search box -- gives an accessible,
# non-3D, filterable view of the same data. Self-contained HTML, no
# external 3D software required.
#
# The terrain color ramp below was checked with computed OKLCH values
# (lightness, chroma, hue -- see project chat history), not eyeballed:
# it fixed two real issues found that way -- a lightness spike at the
# "lightly visited" tan stop that broke monotonic brightness (confusing
# for colorblind/grayscale reading), and deep ocean blue sitting so
# close to the black page background (contrast ~1.7:1) that the map's
# edge was hard to make out. A thin frame around the map extent gives a
# second, color-independent cue for where the map ends. The recency
# ramp (Plotly's built-in "Plasma") was deliberately chosen to sit far
# from the terrain's green/tan/brown hues, so the two color encodings
# don't get visually confused with each other. Both scales also go
# through a computed colorblindness check further down -- again measured,
# not eyeballed -- without changing how they look to typical color vision.
import shapely
import plotly.graph_objects as go
import plotly.colors as pc
from scipy.ndimage import gaussian_filter, distance_transform_edt

PLOTLY_GRID_RESOLUTION = 600  # Coarser than a print texture, tuned for browser interactivity
PLOTLY_BANDWIDTH_FACTOR = 0.65  # Wider than a "sharp peak" look -> rounded hills
TERRAIN_Z_ASPECT = 0.015  # Vertical exaggeration relative to the map footprint
NOISE_AMPLITUDE = 0.5  # Relative perturbation applied to the density field itself
NOISE_SMOOTHNESS = 0.8  # Gaussian blur (grid cells) for the noise field: lower = tighter, more numerous irregularities
LAND_MASK_BLUR_SIGMA = 0.7  # Grid cells; keeps hills from bleeding far out to sea
OCEAN_MIN, OCEAN_MAX = -0.42, -0.20  # Deep-water -> shallow-coastal blue range
PLAINS_MAX = 0.09  # Ceiling of the unvisited-land color-mottling band
TOP_LABEL_COUNT = 15  # Only the N most-visited cities get an always-on label
OUTPUT_HTML_PATH = "interactive_3d_map.html"


def smoothstep(x: np.ndarray, low: float, high: float) -> np.ndarray:
  """Smooth 0 -> 1 ramp with zero slope at both ends (3t^2 - 2t^3), used
  to taper terrain into the flat plain without a hard edge."""
  t = np.clip((x - low) / (high - low), 0.0, 1.0)
  return t * t * (3 - 2 * t)


# --- Elevation surface ---------------------------------------------------
# Reuses every visit point collected in the city-identification step
# above (not just the 54 principal cities), evaluated on a grid sized
# for interactive rendering. Height reflects visit density only -- how
# much time/how many visits a place got -- not how recently it was
# visited; recency is layered on separately via marker color below.
visit_lon = visits_df["longitude"].to_numpy()
visit_lat = visits_df["latitude"].to_numpy()

elevation_kde = gaussian_kde(np.vstack([visit_lon, visit_lat]))
elevation_kde.set_bandwidth(bw_method=elevation_kde.factor * PLOTLY_BANDWIDTH_FACTOR)

lat_span_ratio = (MAX_LAT - MIN_LAT) / (MAX_LON - MIN_LON)
plotly_lon_grid = np.linspace(MIN_LON, MAX_LON, PLOTLY_GRID_RESOLUTION)
plotly_lat_grid = np.linspace(
    MIN_LAT, MAX_LAT, int(PLOTLY_GRID_RESOLUTION * lat_span_ratio)
)
plotly_mesh_lon, plotly_mesh_lat = np.meshgrid(plotly_lon_grid, plotly_lat_grid)
plotly_grid_positions = np.vstack([plotly_mesh_lon.ravel(), plotly_mesh_lat.ravel()])

print("Evaluating elevation KDE on the interactive grid...")
elevation_raw = elevation_kde(plotly_grid_positions).reshape(plotly_mesh_lon.shape)

# Two-stage power-law (gamma) compression: visit density is heavily
# skewed toward home/frequent locations, so a linear scale would leave
# smaller cities nearly invisible next to the dominant peak. Each pass
# still maps 0 -> 0 and 1 -> 1, so true background is unaffected.
DENSITY_GAMMA = 0.4
SECONDARY_GAMMA = 0.2
elevation_compressed = np.power(elevation_raw, DENSITY_GAMMA)
elevation_unit = (elevation_compressed - elevation_compressed.min()) / (
    elevation_compressed.max() - elevation_compressed.min()
)

# Multiplicative noise applied to the density field itself (not just the
# final height), so it perturbs *where* each hill's edge falls -- a
# tight, high-frequency noise field wobbles the boundary into an
# irregular, natural-looking outline instead of a smooth Gaussian oval.
# Multiplicative (not additive) so true background stays exactly flat:
# 0 * (1 + noise) is still 0.
noise_rng = np.random.default_rng(42)  # fixed seed for reproducible terrain
raw_noise = noise_rng.normal(size=elevation_unit.shape)
smooth_noise = gaussian_filter(raw_noise, sigma=NOISE_SMOOTHNESS)
smooth_noise /= np.abs(smooth_noise).max()  # normalize to [-1, 1]
elevation_unit_noisy = np.clip(elevation_unit * (1 + NOISE_AMPLITUDE * smooth_noise), 0.0, 1.0)

elevation_unit2 = np.power(elevation_unit_noisy, SECONDARY_GAMMA)

# Smooth (zero-slope) transition instead of a hard cutoff, so hills taper
# gently into the plain rather than ending in a hard edge. Built from the
# *noisy* field above so the transition boundary itself is irregular.
transition_gate = smoothstep(elevation_unit_noisy, low=0.005, high=0.05)
elevation_hills = elevation_unit2 * transition_gate

# --- Land vs. ocean classification (for hypsometric coloring) -----------
# Uses the 50m-resolution Natural Earth boundaries (much more accurate
# coastlines than the 110m set used elsewhere) -- at 110m, narrow
# bays/inlets like Tampa Bay get smoothed away entirely and read as
# solid land, letting hills bleed visibly out over real open water.
print("Loading world boundaries and classifying grid cells as land/ocean...")
WORLD_URL = "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip"
world = gpd.read_file(WORLD_URL).clip([MIN_LON, MIN_LAT, MAX_LON, MAX_LAT])
land_union = world.union_all()
is_land = shapely.contains_xy(land_union, plotly_mesh_lon, plotly_mesh_lat)

# Blur the land/ocean boolean mask into a smooth 0-1 field and use it to
# constrain elevation itself (not just color), so hills can't rise out
# over open ocean or spill past the real coastline -- a small blur keeps
# the coastline from becoming another hard cliff.
land_mask_smooth = gaussian_filter(is_land.astype(float), sigma=LAND_MASK_BLUR_SIGMA)
elevation_z = np.clip(elevation_hills * land_mask_smooth, 0.0, None)

# --- Background texture: ocean depth + plains mottling -------------------
# Real-world relief maps don't paint the "empty" areas as flat solid
# colors either -- oceans get lighter near the coast and darker with
# depth/distance from land, and plains show subtle natural variation.
# This is color-only: it never touches elevation_z, so hill geometry
# still reflects visit density alone.
print("Shading ocean depth and plains texture...")
distance_from_coast = distance_transform_edt(~is_land)  # in grid cells
shallow_amount = 1.0 - np.clip(distance_from_coast / 40.0, 0.0, 1.0)  # 1 at the coast, 0 far out

background_noise_rng = np.random.default_rng(7)
raw_background_noise = background_noise_rng.normal(size=elevation_z.shape)
background_noise = gaussian_filter(raw_background_noise, sigma=6.0)
background_noise = (background_noise - background_noise.min()) / (
    background_noise.max() - background_noise.min()
)  # normalize to [0, 1]

ocean_depth_mix = 0.6 * shallow_amount + 0.4 * background_noise
ocean_color_value = OCEAN_MIN + (OCEAN_MAX - OCEAN_MIN) * ocean_depth_mix
plains_color_value = PLAINS_MAX * background_noise

# Ocean cells are always painted from the depth-shaded ocean range, full
# stop -- no blend with the terrain ramp. Earlier this blended toward
# green near any coastal city's KDE footprint (e.g. Tampa, Panama City),
# because that blend used the *pre-land-mask* transition_gate: even
# though elevation itself was already correctly suppressed near 0
# offshore, the blend still leaned toward the "0 elevation = green
# plains" color instead of blue. Since land_mask_smooth already tapers
# height smoothly right at the coast, a hard color cutoff here no longer
# clips a tall peak -- only a near-zero sliver -- so there's no
# hard-edge look to trade off. Land cells take whichever is taller: the
# real hill color, or the subtle background mottling on flat ground.
terrain_value = np.where(
    is_land, np.maximum(elevation_z, plains_color_value), ocean_color_value
)


def color_stop(value: float, color: str) -> list:
  """Converts a raw terrain_value into a normalized Plotly colorscale stop."""
  return [(value - OCEAN_MIN) / (1.0 - OCEAN_MIN), color]


# Hypsometric ("real relief map") color ramp: deep ocean -> shallow
# coastal water -> mottled plains -> hills -> mountains -> snow-capped
# peaks. "#124d7a" (deep ocean) and "#b0a24e" (the 0.46 stop) replace
# darker/lighter originals per the OKLCH check noted above.
TERRAIN_COLORSCALE = [
    color_stop(OCEAN_MIN, "#124d7a"),
    color_stop(OCEAN_MAX, "#1c5f8c"),
    color_stop(0.0, "#3f6b35"),
    color_stop(PLAINS_MAX, "#5c8f4a"),
    color_stop(0.30, "#7fa84a"),
    color_stop(0.46, "#b0a24e"),
    color_stop(0.55, "#c99a4a"),
    color_stop(0.65, "#a66a35"),
    color_stop(0.80, "#7a5230"),
    color_stop(0.92, "#9a9a9a"),
    color_stop(1.0, "#ffffff"),
]

# --- Computed colorblindness (CVD) validation -----------------------------
# Simulates protanopia/deuteranopia/tritanopia (Machado, Oliveira &
# Fernandes, 2009, severity 1.0) on both color scales and measures
# perceptual distance (OKLab delta-E x100) between ADJACENT stops --
# computed, not eyeballed, matching how the OKLCH lightness/contrast
# fixes above were originally found. This is a read-only check: nothing
# here changes what people with typical color vision see, it only
# documents where the simulated colors get harder to tell apart, and
# whether that's already mitigated elsewhere.
_CVD_MATRICES = {  # applied directly to linear RGB
    "protanopia": [[0.152286, 1.052583, -0.204868],
                   [0.114503, 0.786281, 0.099216],
                   [-0.003882, -0.048116, 1.051998]],
    "deuteranopia": [[0.367322, 0.860646, -0.227968],
                      [0.280085, 0.672501, 0.047413],
                      [-0.011820, 0.042940, 0.968881]],
    "tritanopia": [[1.255528, -0.076749, -0.178779],
                    [-0.078411, 0.930809, 0.147602],
                    [0.004733, 0.691367, 0.303900]],
}


def _srgb_255_to_linear(channel: float) -> float:
  c = channel / 255.0
  return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4


def _linear_to_oklab(r: float, g: float, b: float) -> tuple:
  l = 0.4122214708 * r + 0.5363325363 * g + 0.0514459929 * b
  m = 0.2119034982 * r + 0.6806995451 * g + 0.1073969566 * b
  s = 0.0883024619 * r + 0.2817188376 * g + 0.6299787005 * b
  l, m, s = l ** (1 / 3), m ** (1 / 3), s ** (1 / 3)
  return (
      0.2104542553 * l + 0.7936177850 * m - 0.0040720468 * s,
      1.9779984951 * l - 2.4285922050 * m + 0.4505937099 * s,
      0.0259040371 * l + 0.7827717662 * m - 0.8086757660 * s,
  )


def _color_to_linear_rgb(color: str) -> tuple:
  """Accepts '#rrggbb' hex or Plotly's 'rgb(r, g, b)' strings."""
  if color.startswith("#"):
    channels = (int(color[i:i + 2], 16) for i in (1, 3, 5))
  else:
    channels = (float(v) for v in color.strip("rgb()").split(","))
  return tuple(_srgb_255_to_linear(c) for c in channels)


def worst_adjacent_cvd_delta_e(stops: list) -> dict:
  """For each simulated deficiency, the smallest OKLab delta-E (x100)
  between any two ADJACENT stops -- the pair most likely to be confused
  with each other under that type of color blindness."""
  linear_stops = [_color_to_linear_rgb(c) for c in stops]
  worst_per_kind = {}
  for cvd_name, matrix in _CVD_MATRICES.items():
    lab_points = []
    for r, g, b in linear_stops:
      simulated = [max(0.0, min(1.0, sum(m * c for m, c in zip(row, (r, g, b)))))
                   for row in matrix]
      lab_points.append(_linear_to_oklab(*simulated))
    worst_per_kind[cvd_name] = min(
        100 * np.linalg.norm(np.subtract(lab_points[i], lab_points[i + 1]))
        for i in range(len(lab_points) - 1)
    )
  return worst_per_kind


_terrain_stops_hex = [color for _, color in TERRAIN_COLORSCALE]
_recency_stops_rgb = pc.sample_colorscale("Plasma", [i / 7 for i in range(8)])

print("Colorblindness (CVD) validation -- worst adjacent OKLab delta-E x100 "
      "(pairs stay easily distinguishable at >= 8, per the project's usual palette check):")
print("  Terrain colorscale:",
      {k: round(v, 1) for k, v in worst_adjacent_cvd_delta_e(_terrain_stops_hex).items()})
print("  Recency (Plasma):  ",
      {k: round(v, 1) for k, v in worst_adjacent_cvd_delta_e(_recency_stops_rgb).items()})
print(
    "  Terrain's weakest pair falls in the green-to-khaki transition -- a "
    "known hard spot for red-green CVD in any hypsometric map. Mitigated "
    "already: elevation is real 3D height + shading, not color alone, so "
    "relative intensity still reads by geometry. Recency's weakest pair is "
    "still separable by lightness (the legend already reads 'dark = long "
    "ago, bright = recent'), and the exact date is also in the hover text "
    "and the table below. No colors were changed here, so the map looks "
    "identical for typical color vision."
)


def sample_elevation_grid(lat_values, lon_values) -> np.ndarray:
  """Vectorized nearest-grid-cell elevation lookup for arrays of lat/lon."""
  lat_values = np.asarray(lat_values, dtype=float)
  lon_values = np.asarray(lon_values, dtype=float)
  col_idx = np.clip(
      np.searchsorted(plotly_lon_grid, lon_values), 0, len(plotly_lon_grid) - 1
  )
  row_idx = np.clip(
      np.searchsorted(plotly_lat_grid, lat_values), 0, len(plotly_lat_grid) - 1
  )
  return elevation_z[row_idx, col_idx]


# --- Country borders, draped over the terrain ----------------------------
def boundary_to_scatter3d_coords(boundary_geoseries, z_offset: float = 0.01):
  """Flattens a GeoSeries of (Multi)LineString geometries into flat x/y/z
  coordinate lists for a single Plotly Scatter3d line trace. Each point
  is draped at the local terrain height (+ a small offset so the line
  stays visibly above the surface instead of being buried in a hill),
  with None breaks between disconnected segments."""
  xs, ys, zs = [], [], []
  for geom in boundary_geoseries:
    if geom is None or geom.is_empty:
      continue
    lines = list(geom.geoms) if geom.geom_type == "MultiLineString" else [geom]
    for line in lines:
      lon, lat = np.asarray(line.xy[0]), np.asarray(line.xy[1])
      z = sample_elevation_grid(lat, lon) + z_offset
      xs.extend(lon); xs.append(None)
      ys.extend(lat); ys.append(None)
      zs.extend(z); zs.append(None)
  return xs, ys, zs


border_x, border_y, border_z = boundary_to_scatter3d_coords(world.boundary)

# Thin neatline around the full map extent -- a second, color-independent
# cue for where the map ends, since deep ocean blue alone is still fairly
# dark against the black page background.
frame_x = [MIN_LON, MAX_LON, MAX_LON, MIN_LON, MIN_LON]
frame_y = [MIN_LAT, MIN_LAT, MAX_LAT, MAX_LAT, MIN_LAT]
frame_z = [0.001] * 5

# --- City markers & labels -------------------------------------------------
visited_cities_df["marker_z"] = sample_elevation_grid(
    visited_cities_df["latitude"], visited_cities_df["longitude"]
) + 0.02

# Label + hover text share the same simple format: "City - <year of last
# visit>". Kept deliberately terse -- the fuller detail (state, country,
# exact date, visit count) lives in the accessible table below instead of
# cluttering every hover.
visited_cities_df["label_text"] = (
    visited_cities_df["city"] + " - "
    + visited_cities_df["last_visit"].dt.year.astype(str)
)

# Recency, normalized across all principal cities into 0 (oldest last
# visit) .. 1 (most recent), drives marker color -- a second, independent
# encoding layered on top of visit-count-driven terrain height/color.
last_visit_numeric = visited_cities_df["last_visit"].astype("int64")
visited_cities_df["recency_norm"] = (
    (last_visit_numeric - last_visit_numeric.min())
    / (last_visit_numeric.max() - last_visit_numeric.min())
)

# Labeling all 54 principal cities on the map at once makes the dense
# clusters (e.g. greater Bogota, Florida) unreadable at the default zoom.
# Only the most-visited TOP_LABEL_COUNT get an always-on label; the rest
# still get a marker and a hover tooltip, and read fine once you zoom
# into their area. Unlabeled terrain elsewhere on the map is visited too
# -- it just didn't cluster into one of these 54 principal cities.
labeled_cities_df = visited_cities_df.nlargest(TOP_LABEL_COUNT, "visit_count")
other_cities_df = visited_cities_df.drop(labeled_cities_df.index)

# --- Camera presets --------------------------------------------------------
# Plotly maps a manual-aspectratio scene's full data range onto a cube
# centered at (0, 0, 0) and spanning [-aspectratio/2, +aspectratio/2] per
# axis; scene.camera.eye/center are expressed in that normalized space, not
# raw lon/lat. scene_camera_for() converts a target lon/lat into it, so a
# button can frame a specific place without hardcoding normalized numbers.
def scene_camera_for(lon: float, lat: float, y_offset: float = 0.22, z_offset: float = 0.16) -> dict:
  lon_mid = (MIN_LON + MAX_LON) / 2.0
  lat_mid = (MIN_LAT + MAX_LAT) / 2.0
  center_x = (lon - lon_mid) / (MAX_LON - MIN_LON)
  center_y = (lat - lat_mid) / (MAX_LAT - MIN_LAT) * lat_span_ratio
  return dict(
      eye=dict(x=center_x, y=center_y - y_offset, z=z_offset),
      center=dict(x=center_x, y=center_y, z=0),
      up=dict(x=0, y=0, z=1),
  )


DEFAULT_CAMERA = dict(eye=dict(x=0.0, y=-0.95, z=0.65), center=dict(x=0, y=0, z=0), up=dict(x=0, y=0, z=1))
TOP_VIEW_CAMERA = dict(eye=dict(x=0.0001, y=0.0, z=1.3), center=dict(x=0, y=0, z=0), up=dict(x=0, y=1, z=0))
most_visited_city = visited_cities_df.nlargest(1, "visit_count").iloc[0]
MY_AREA_CAMERA = scene_camera_for(most_visited_city["longitude"], most_visited_city["latitude"])

# --- Assemble the figure ---------------------------------------------------
fig = go.Figure()

fig.add_trace(go.Surface(
    x=plotly_mesh_lon, y=plotly_mesh_lat, z=elevation_z,
    surfacecolor=terrain_value,
    colorscale=TERRAIN_COLORSCALE,
    cmin=OCEAN_MIN, cmax=1.0,
    showscale=True,
    # Custom tick labels turn the raw internal color scale into a plain-
    # language legend instead of showing the underlying numeric range.
    colorbar=dict(
        tickvals=[-0.35, 0.0, 0.3, 0.6, 1.0],
        ticktext=[
            "Ocean", "Unvisited", "Lightly visited",
            "Frequently visited", "Most visited",
        ],
        tickfont=dict(color="white", size=11),
        thickness=16, len=0.45, x=1.0, y=0.4, yanchor="middle",
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,255,255,0.25)", borderwidth=1,
        outlinewidth=0,
    ),
    lighting=dict(ambient=0.55, diffuse=0.75, specular=0.15, roughness=0.9),
    hoverinfo="skip",
))

fig.add_trace(go.Scatter3d(
    x=border_x, y=border_y, z=border_z,
    mode="lines",
    line=dict(color="rgba(15,15,15,0.85)", width=2.5),
    hoverinfo="skip",
    showlegend=False,
))

fig.add_trace(go.Scatter3d(
    x=frame_x, y=frame_y, z=frame_z,
    mode="lines",
    line=dict(color="rgba(220,225,235,0.35)", width=2),
    hoverinfo="skip",
    showlegend=False,
))

# Top cities: marker + always-on label. Marker fill color encodes
# recency via the shared "coloraxis" (defined in update_layout below) so
# both marker traces plot on the same scale and share a single legend; a
# white outline keeps every dot visible regardless of fill color or
# underlying terrain shade.
fig.add_trace(go.Scatter3d(
    x=labeled_cities_df["longitude"], y=labeled_cities_df["latitude"],
    z=labeled_cities_df["marker_z"],
    mode="markers+text",
    marker=dict(
        size=3.2, color=labeled_cities_df["recency_norm"], coloraxis="coloraxis",
        symbol="circle", line=dict(color="white", width=0.6),
    ),
    text=labeled_cities_df["label_text"],
    textposition="top center",
    textfont=dict(color="white", size=9, family="Arial, sans-serif"),
    hovertext=labeled_cities_df["label_text"],
    hoverinfo="text",
    showlegend=False,
))

# Remaining cities: marker + hover only, no permanent label
fig.add_trace(go.Scatter3d(
    x=other_cities_df["longitude"], y=other_cities_df["latitude"],
    z=other_cities_df["marker_z"],
    mode="markers",
    marker=dict(
        size=3.2, color=other_cities_df["recency_norm"], coloraxis="coloraxis",
        symbol="circle", line=dict(color="white", width=0.6),
    ),
    hovertext=other_cities_df["label_text"],
    hoverinfo="text",
    showlegend=False,
))

fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        aspectmode="manual",
        aspectratio=dict(x=1, y=lat_span_ratio, z=TERRAIN_Z_ASPECT),
        camera=DEFAULT_CAMERA,
        bgcolor="black",
    ),
    # Shared color axis for both marker traces above: "Plasma" (dark
    # purple -> bright yellow) sits far from the terrain's green/tan/
    # brown hues so the two color encodings read as clearly separate.
    coloraxis=dict(
        colorscale="Plasma", cmin=0, cmax=1,
        colorbar=dict(
            title=dict(text="Last visit", font=dict(color="white", size=12)),
            tickvals=[0, 1], ticktext=["Older", "Recent"],
            tickfont=dict(color="white", size=10),
            thickness=14, len=0.22, x=1.0, y=0.90, yanchor="middle",
            bgcolor="rgba(0,0,0,0.35)",
            bordercolor="rgba(255,255,255,0.25)", borderwidth=1,
            outlinewidth=0,
        ),
    ),
    paper_bgcolor="black",
    margin=dict(l=0, r=0, t=70, b=0),
    showlegend=False,
    # Title
    title=dict(
        text=(
            "<b>My Google Timeline in 3D</b>"
            "<br><span style='font-size:13px;color:#aaaaaa'>"
            "Places visited across the Americas</span>"
        ),
        x=0.02, xanchor="left", y=0.97, yanchor="top",
        font=dict(color="white", size=24, family="Arial, sans-serif"),
    ),
    # Camera-preset buttons: quicker, more discoverable ways to explore
    # than dragging alone, without replacing free rotate/zoom/pan.
    updatemenus=[
        dict(
            type="buttons", direction="right", showactive=False,
            buttons=[
                dict(label="Isometric", method="relayout", args=["scene.camera", DEFAULT_CAMERA]),
                dict(label="Top view", method="relayout", args=["scene.camera", TOP_VIEW_CAMERA]),
                dict(
                    label=f"Most visited ({most_visited_city['city']})",
                    method="relayout", args=["scene.camera", MY_AREA_CAMERA],
                ),
            ],
            x=0.02, xanchor="left", y=0.90, yanchor="top",
            bgcolor="rgba(0,0,0,0.55)", bordercolor="rgba(255,255,255,0.3)", borderwidth=1,
            font=dict(color="white", size=11),
            pad=dict(l=8, r=8, t=4, b=4),
        ),
    ],
    # Color-scale caption + a short "how to read this map" note. Both are
    # anchored to the page (paper coordinates), so they stay fixed on
    # screen while the terrain rotates/zooms underneath. Plain Unicode
    # punctuation ("&" and "·") is used directly instead of HTML
    # entities -- Plotly's text renderer doesn't expand "&middot;", so it
    # was showing up literally instead of as a dot.
    annotations=[
        dict(
            text="<b>Visit intensity</b>",
            xref="paper", yref="paper",
            x=1.0, y=0.63, xanchor="center", yanchor="bottom",
            showarrow=False, font=dict(color="white", size=12),
        ),
        dict(
            text=(
                "<b>How to read this map</b><br>"
                "Terrain height & color = Visit frequency · "
                "Dot color = Recency (Dark = Long ago, Bright = Recent)<br>"
                f"Top {TOP_LABEL_COUNT} cities labeled; Hover any dot for its "
                "name · Unlabeled peaks are visited too<br>"
                "Drag to rotate · Scroll to zoom · Toolbar reset "
                "icon or double-click resets the view"
            ),
            xref="paper", yref="paper",
            x=0.02, y=0.02, xanchor="left", yanchor="bottom",
            align="left", showarrow=False, font=dict(color="white", size=12),
            bgcolor="rgba(0,0,0,0.45)",
            bordercolor="rgba(255,255,255,0.25)", borderwidth=1, borderpad=8,
        ),
    ],
)

# --- Accessible table view ---------------------------------------------
# A plain HTML <table> (not a Plotly widget) beneath the 3D map: readable
# by screen readers, browser find-in-page, and anyone whose browser can't
# render WebGL -- a fallback the 3D view alone can't provide. A small
# vanilla-JS search box gives a fast, reliable way to find/jump to a
# specific city (a 3D camera-jump control for 54 points would be far
# more fragile than filtering rows of an already-accessible table).
table_source_df = (
    visited_cities_df[["city", "state", "country", "last_visit", "visit_count"]]
    .sort_values("visit_count", ascending=False)
    .reset_index(drop=True)
)
table_source_df["last_visit"] = table_source_df["last_visit"].dt.strftime("%Y-%m-%d")
table_source_df.columns = ["City", "State/Region", "Country", "Last Visit", "Visits"]
cities_table_html = table_source_df.to_html(
    index=False, border=0, classes="cities-table", table_id="cities-table"
)

# Keep the toolbar (with its built-in "reset camera to default" home
# icon) visible, and drop the Plotly logo button for a cleaner look.
# div_id is set explicitly so the loading-overlay script below can find
# this exact plot without guessing Plotly's auto-generated id.
plot_div_html = fig.to_html(
    full_html=False, include_plotlyjs="cdn",
    config=dict(displayModeBar=True, displaylogo=False),
    div_id="plotly-map",
)

full_page_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>My Google Timeline in 3D</title>
<style>
  body {{ background: #000; color: #e5e5e5; font-family: Arial, sans-serif; margin: 0; }}
  section.table-view {{ max-width: 900px; margin: 0 auto; padding: 24px 16px 64px; }}
  section.table-view h2 {{ color: #fff; font-size: 18px; }}
  input#city-search {{
    width: 100%; box-sizing: border-box; margin: 12px 0 16px; padding: 8px 12px;
    background: #141414; color: #e5e5e5; border: 1px solid #444; border-radius: 4px;
    font-size: 14px;
  }}
  input#city-search:focus {{ outline: none; border-color: #888; }}
  table.cities-table {{ width: 100%; border-collapse: collapse; font-size: 14px; }}
  table.cities-table th, table.cities-table td {{
    text-align: left; padding: 6px 10px; border-bottom: 1px solid #333;
  }}
  table.cities-table th {{ color: #fff; border-bottom: 2px solid #555; }}
  table.cities-table tr:hover {{ background: #141414; }}
  p#city-search-empty {{ display: none; color: #999; font-size: 13px; padding: 8px 10px; }}
  /* Covers the page while plotly.js downloads and builds the WebGL scene
     -- the file is large (tens of MB with the full dataset embedded), so
     the map can take a moment to become interactive. Resolution/detail
     is unchanged; this only affects perceived load time. */
  #loading-overlay {{
    position: fixed; inset: 0; z-index: 9999;
    display: flex; flex-direction: column; align-items: center; justify-content: center;
    gap: 14px; background: #000;
  }}
  #loading-overlay .spinner {{
    width: 42px; height: 42px; border-radius: 50%;
    border: 4px solid rgba(255,255,255,0.15); border-top-color: #fff;
    animation: spin 0.9s linear infinite;
  }}
  #loading-overlay p {{ color: #bbb; font-size: 14px; margin: 0; }}
  @keyframes spin {{ to {{ transform: rotate(360deg); }} }}
</style>
</head>
<body>
<div id="loading-overlay">
  <div class="spinner"></div>
  <p>Loading 3D map&hellip;</p>
</div>
{plot_div_html}
<section class="table-view">
  <h2>Visited cities (table view)</h2>
  <p style="color:#999; font-size:13px;">
    Accessible, non-3D view of the same {len(visited_cities_df)} principal cities
    shown on the map above.
  </p>
  <input type="text" id="city-search" placeholder="Search a city, state, or country&hellip;">
  {cities_table_html}
  <p id="city-search-empty">No cities match your search.</p>
</section>
<script>
(function() {{
  var searchInput = document.getElementById("city-search");
  var table = document.getElementById("cities-table");
  var emptyMessage = document.getElementById("city-search-empty");
  if (!searchInput || !table) return;
  var rows = table.querySelectorAll("tbody tr");
  searchInput.addEventListener("input", function() {{
    var query = searchInput.value.trim().toLowerCase();
    var visibleCount = 0;
    rows.forEach(function(row) {{
      var matches = row.textContent.toLowerCase().indexOf(query) !== -1;
      row.style.display = matches ? "" : "none";
      if (matches) visibleCount += 1;
    }});
    emptyMessage.style.display = visibleCount === 0 ? "block" : "none";
  }});
}})();
(function() {{
  var overlay = document.getElementById("loading-overlay");
  var mapDiv = document.getElementById("plotly-map");
  function hideOverlay() {{
    if (overlay) overlay.style.display = "none";
  }}
  if (mapDiv && mapDiv.on) {{
    mapDiv.on("plotly_afterplot", hideOverlay);
  }}
  window.setTimeout(hideOverlay, 8000);  // safety net if the event never fires
}})();
</script>
</body>
</html>"""

with open(OUTPUT_HTML_PATH, "w", encoding="utf-8") as file:
  file.write(full_page_html)

print(f"✅ Interactive 3D map + table saved to {OUTPUT_HTML_PATH}")
fig.show()